# Training Device: Twister / harvesting

Adapted from quickstart.ipynb for the training computer. Edit Configuration, then Run All.
**Cells only print PowerShell commands:** paste selected commands into that computer's terminal.
Run All does not install packages, connect hardware, upload data or start training/inference.
Select the harvesting_train kernel after registering it. Any Python kernel can generate initial setup commands.
Skip installations and environment creation that are already complete.
Use matching embodiment, dataset/model IDs and Git commit on both computers.

In [ ]:
import json
from pathlib import Path

def find_repo(default: str) -> str:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/lerobot").is_dir():
            return str(candidate)
    return default

def ps_quote(value: str) -> str:
    return "'" + str(value).replace("'", "''") + "'"

def flag(name: str, value) -> str:
    if isinstance(value, bool): text = str(value).lower()
    elif isinstance(value, dict): text = json.dumps(value)
    else: text = str(value)
    return f"--{name}={ps_quote(text)}"

def print_cmd(*parts: str) -> None:
    """Generate PowerShell commands; never execute them."""
    print(" `\n  ".join(part for part in parts if part))

def print_session(environment: str, repo_path: str) -> None:
    print(f"conda activate {environment}")
    print(f"Set-Location -LiteralPath {ps_quote(repo_path)}")
    print('$env:PYTHONUTF8 = "1"')

## Configuration
Choose bimanual_harvesting (Twister + SO-100) or twister; match the hardware notebook.
Replace YOUR_HF_USERNAME. Use a new RUN_NAME for a fresh run; resume uses the saved configuration.

In [ ]:
ENV_NAME = "harvesting_train"
REPO_PATH = find_repo(r"D:\FYP_HS\lerobotharvesting1\lerobot_harvesting1")
EMBODIMENT = "bimanual_harvesting"  # Or "twister".
assert EMBODIMENT in {"twister", "bimanual_harvesting"}
HF_USER = "YOUR_HF_USERNAME"
DATASET_NAME = f"{EMBODIMENT}_demo"
POLICY_NAME = f"{EMBODIMENT}_act"
DATASET_REPO_ID = f"{HF_USER}/{DATASET_NAME}"
POLICY_REPO_ID = f"{HF_USER}/{POLICY_NAME}"
POLICY_DEVICE = "cuda"
VIDEO_BACKEND = "pyav"
DATASET_LOCAL_ROOT = None  # Set to copied dataset directory for a local-only transfer.
TRAIN_STEPS = 20_000
SAVE_FREQ = 2_000
BATCH_SIZE = 8
RUN_NAME = f"{EMBODIMENT}_act"
OUTPUT_DIR = f"outputs/train/{RUN_NAME}"
PUSH_MODEL_TO_HUB = True
LAST_CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoints/last"
PRETRAINED_MODEL_PATH = f"{LAST_CHECKPOINT_PATH}/pretrained_model"
TRAIN_CONFIG_PATH = f"{PRETRAINED_MODEL_PATH}/train_config.json"
print(f"Dataset: {DATASET_REPO_ID}\nPolicy: {POLICY_REPO_ID}\nOutput: {OUTPUT_DIR}")
print("Replace YOUR_HF_USERNAME before running terminal commands.")

## 1. Initialise once; activate each terminal session
You already created harvesting_train, installed .[training] and installed FFmpeg: skip those steps if complete.
The Conda environment stays local and outside the shared repo. ipykernel enables selection in VS Code.
Standard PyAV wheels bundle FFmpeg libraries; the explicit FFmpeg install is compatible but optional for PyAV.

In [ ]:
print(f"Set-Location -LiteralPath {ps_quote(REPO_PATH)}")
print(f"conda create -n {ENV_NAME} python=3.12 -y")
print_session(ENV_NAME, REPO_PATH)
print('python -m pip install -e ".[training]"')
print("conda install -c conda-forge ffmpeg")
print("python -m pip install ipykernel")
print(f'python -m ipykernel install --user --name {ENV_NAME} --display-name "{ENV_NAME} (Python 3.12)"')
print("\n# Later terminal sessions:")
print_session(ENV_NAME, REPO_PATH)

## 2. Verify interpreter and GPU
Run these optional diagnostics in harvesting_train. The LeRobot path should point to this repo.
Require Available: True and a successful CUDA tensor computation before CUDA training.

In [ ]:
print("python -c " + ps_quote("import sys,lerobot; print(sys.executable); print(lerobot.__file__)"))
print("python -c " + ps_quote("import torch; print('Torch:',torch.__version__); print('CUDA:',torch.version.cuda); print('Available:',torch.cuda.is_available())"))
print("python -c " + ps_quote("import torch; print(torch.cuda.get_device_name(0)); print(torch.rand(100,device='cuda').sum().item())"))
print("python -m pip check")

### Conditional repair for the diagnosed CPU-only PyTorch
Skip if CUDA already works. This PC was diagnosed with torch 2.11.0+cpu and RTX 4080 SUPER,
NVIDIA driver 561.03 (CUDA 12.6 support). These pinned versions match that driver and this repo:
PyTorch 2.8.0 / torchvision 0.23.0 / TorchCodec 0.7.0. Re-run the GPU checks afterwards.
This is specific to the training computer, not a universal configuration for the hardware computer.

In [ ]:
print_session(ENV_NAME, REPO_PATH)
print("python -m pip install --no-deps torchcodec==0.7.0")
print("python -m pip install --upgrade torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu126")
print("python -m pip check")

## 3. Access the uploaded dataset
Wait for hardware recording/upload to finish. Login locally with dataset access and model write access.
For manual dataset transfer, copy the complete dataset and set DATASET_LOCAL_ROOT to its destination here.
Do not store authentication tokens in notebooks or Git.

In [ ]:
print("hf auth login")
print("hf auth whoami")
print(f"Dataset: https://huggingface.co/datasets/{DATASET_REPO_ID}")

## 4. Train ACT and upload the final policy
Training uses dataset features and needs no serial ports or calibration. These are starting experiment
settings; two recording-check episodes do not provide a useful training dataset by themselves.
Choose a new output directory for each fresh run. Uploading the final policy and processors makes the
model available to the hardware computer. Keep the repo commit consistent throughout a run.

In [ ]:
print_cmd(
    "lerobot-train", flag("dataset.repo_id", DATASET_REPO_ID),
    flag("dataset.root", DATASET_LOCAL_ROOT) if DATASET_LOCAL_ROOT else "",
    flag("dataset.video_backend", VIDEO_BACKEND), flag("policy.type", "act"),
    flag("policy.device", POLICY_DEVICE), flag("policy.repo_id", POLICY_REPO_ID),
    flag("policy.push_to_hub", PUSH_MODEL_TO_HUB), flag("output_dir", OUTPUT_DIR),
    flag("job_name", RUN_NAME), flag("batch_size", BATCH_SIZE), flag("steps", TRAIN_STEPS),
    flag("save_freq", SAVE_FREQ), flag("env_eval_freq", 0), flag("wandb.enable", False),
)

## 5. Optional: resume interrupted training
Use this instead of a fresh training command, after a checkpoint exists. The saved configuration
restores dataset, policy and optimizer settings. This path is local to the training computer.

In [ ]:
print_cmd("lerobot-train", flag("config_path", TRAIN_CONFIG_PATH), flag("resume", True))

## 6. Handoff to the hardware computer
After upload, set POLICY_PATH in hardware_device.ipynb to the model repository ID printed below.
The hardware computer runs inference locally; this training computer need not stay running.
If upload is disabled, copy the entire pretrained_model folder (weights, config and processors) to the
hardware computer and set POLICY_PATH to that local destination. Git source sharing does not copy artifacts.

Both computers create local Conda environments and authenticate locally. Do not sync .venv, caches or
generated outputs. Use distinct output directories if the repo is on a shared network filesystem.
See ../../docs/harvesting/USE_CASE_CHANGELOG.md for the file inventory.

In [ ]:
print(f"Hardware POLICY_PATH = {POLICY_REPO_ID!r}")
print(f"Local directory for manual model transfer: {PRETRAINED_MODEL_PATH}")
print(f"Recorded embodiment must be: {EMBODIMENT}")